In [1]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

print(dataset)

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
})


In [4]:
def create_sentiment_label(stars):
    if stars <= 2:
        return "negative"
    elif stars == 3:
        return "neutral"
    else:
        return "positive"


def prepare_split(split):
    df = split.to_pandas()
    df["sentiment"] = df["stars"].apply(create_sentiment_label)
    return df


train_df = prepare_split(dataset["train"])
val_df = prepare_split(dataset["validation"])
test_df = prepare_split(dataset["test"])

In [5]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [6]:
train_df_clean = train_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

val_df_clean = val_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

test_df_clean = test_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

train_df_clean["text"] = train_df_clean["review_body"].apply(clean_text)
val_df_clean["text"] = val_df_clean["review_body"].apply(clean_text)
test_df_clean["text"] = test_df_clean["review_body"].apply(clean_text)

print("Train:", train_df_clean.shape)
print("Validation:", val_df_clean.shape)
print("Test:", test_df_clean.shape)

Train: (199499, 10)
Validation: (4999, 10)
Test: (5000, 10)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df_clean["text"])
X_val = vectorizer.transform(val_df_clean["text"])
X_test = vectorizer.transform(test_df_clean["text"])

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)


X_train shape: (199499, 50000)
X_val shape: (4999, 50000)
X_test shape: (5000, 50000)


In [8]:
y_train = train_df_clean["sentiment"]
y_val = val_df_clean["sentiment"]
y_test = test_df_clean["sentiment"]

print(y_train.value_counts())

sentiment
negative    79819
positive    79730
neutral     39950
Name: count, dtype: int64


In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model training completed!")

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Model training completed!


In [13]:
y_val_pred = model.predict(X_val)

print("Validation predictions completed!")

Validation predictions completed!


In [14]:
from sklearn.metrics import accuracy_score, classification_report

val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

Validation Accuracy: 0.7395

Classification Report:
              precision    recall  f1-score   support

    negative       0.74      0.84      0.79      1999
     neutral       0.49      0.30      0.37      1000
    positive       0.81      0.85      0.83      2000

    accuracy                           0.74      4999
   macro avg       0.68      0.67      0.66      4999
weighted avg       0.72      0.74      0.72      4999



In [15]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=["negative", "neutral", "positive"]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Neutral", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Neutral", "Predicted Positive"]
)

cm_df

,Predicted Negative,Predicted Neutral,Predicted Positive
Actual Negative,1687,168,144
Actual Neutral,441,300,259
Actual Positive,140,150,1710


In [16]:
error_analysis = val_df_clean[[
    "review_body",
    "sentiment"
]].copy()

error_analysis["predicted"] = y_val_pred

neutral_errors = error_analysis[
    (error_analysis["sentiment"] == "neutral") &
    (error_analysis["predicted"] != "neutral")
]

neutral_errors.head(20)

,review_body,sentiment,predicted
2002,Good for Mosquito's need something for flies,neutral,positive
2003,Should fit a kids head,neutral,positive
2004,quality is nice. wish it had pockets like the ...,neutral,positive
2005,I bought this for my sister who really likes i...,neutral,positive
2006,If you're also buying the Taylors Fundamentals...,neutral,positive
2007,"It does protect the Macbook, so that is what i...",neutral,negative
2008,Very light and easy to store in a small closet...,neutral,positive
2009,This chapstick is a more expensive packaging w...,neutral,negative
2011,It was kinda a slow movie,neutral,negative
2012,I love it but it's too big.,neutral,positive


In [17]:
y_test_pred = model.predict(X_test)

print("Test predictions completed!")

Test predictions completed!


In [18]:
from sklearn.metrics import accuracy_score, classification_report

test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nTest Classification Report:")
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.7336

Test Classification Report:
              precision    recall  f1-score   support

    negative       0.75      0.84      0.79      2000
     neutral       0.45      0.29      0.35      1000
    positive       0.80      0.85      0.82      2000

    accuracy                           0.73      5000
   macro avg       0.67      0.66      0.66      5000
weighted avg       0.71      0.73      0.72      5000



## Baseline Results — TF-IDF + Logistic Regression

The TF-IDF + Logistic Regression baseline achieved a test accuracy of **73.36%**.

The model performed well on positive and negative reviews:

- Negative F1-score: **0.79**
- Positive F1-score: **0.82**

However, performance on neutral reviews was substantially lower:

- Neutral F1-score: **0.35**

Error analysis showed that neutral reviews were frequently classified as positive or negative, especially when they contained strong sentiment words or mixed opinions.

This provides a strong baseline for comparison with transformer-based models such as BERT.